In [ ]:
import ast

import httpx
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext
from pydantic_ai.capabilities import MCP
from pydantic_ai.messages import ModelRequest, ModelResponse, ToolCallPart, ToolReturnPart
from pydantic_ai.models.openrouter import OpenRouterModel, OpenRouterModelSettings
from pydantic_ai.providers.openrouter import OpenRouterProvider

from setup import *

# Helper Functions fürs Logging

In [ ]:
def log_mcp_usage(response):

    mcp_calls = []

    # 1) Alle ToolCallParts aus den ModelResponses sammeln
    for msg in response.all_messages():
        if isinstance(msg, ModelResponse):
            for part in msg.parts:
                if isinstance(part, ToolCallPart):
                    mcp_calls.append({
                        "tool_call_id": part.tool_call_id,
                        "tool_name": part.tool_name,
                        "args": part.args_as_dict(),
                        "result": None,
                        "outcome": None,
                    })
    returns_by_type = {}
    for call in mcp_calls:
        returns_by_type.setdefault(call["tool_name"], []).append(call)

    mcp_call_counts = {tool_name: len(calls) for tool_name, calls in returns_by_type.items()}
    return mcp_call_counts

In [ ]:
input_price_pMt = 0.09
output_price_pMt = 0.18

In [ ]:
# Logging der Agent-Aktivitäten
def monitor_tokens(response, input_price_pMt=input_price_pMt, output_price_pMt=output_price_pMt):
    
    # Message History auslesen
    if hasattr(response, 'messages_history'):
        messages = response.messages_history
    else:
        messages = response.all_messages() if hasattr(response, 'all_messages') else []
    
    input_tokens = response.usage.input_tokens if response.usage else 'N/A'
    output_tokens = response.usage.output_tokens if response.usage else 'N/A'
    costs = (input_tokens * input_price_pMt + output_tokens * output_price_pMt)/1000000 if response.usage else 'N/A'
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "costs": costs,
        "mcp_usage": log_mcp_usage(response)
    }
    

# Agent inkl MCP Setup

In [ ]:
poliscope_mcp=f'https://api.poliscope.de/v2/mcp/poliscope?access_token={os.getenv("POLISCOPE_API_KEY")}'

In [ ]:
with open('./system_prompt.txt', 'r') as file:
    system_prompt = file.read().replace('\n', '')

In [ ]:
# OpenRouter Konfiguration für Gemini Flash
OPEN_ROUTER_KEY = os.getenv("OPEN_ROUTER_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

model = OpenRouterModel(
    "deepseek/deepseek-v4-flash",
    provider=OpenRouterProvider(api_key=OPEN_ROUTER_KEY),
)
settings = OpenRouterModelSettings(
    openrouter_reasoning={
        'effort': 'high',
    },
    openrouter_usage={
        'include': True,
    }
)
# Agent mit Gemini Flash über OpenRouter
agent = Agent(
    model,
    model_settings=settings,
    system_prompt=system_prompt,
   #capabilities=[MCP(poliscope_mcp)]
)

# Analyse

In [ ]:
data = pd.read_csv("./data/raw/big_cities_heat_planning.csv")

In [ ]:
data["entity_name"] = data["context"].apply(lambda x: ast.literal_eval(x)["entityName"])
data["entity_id"] = data["context"].apply(lambda x: ast.literal_eval(x)["entityId"])
data["date"] = pd.to_datetime(data["date"], format="%Y-%m-%dT%H:%M:%S")
data = data.sort_values(by="date", ascending=False)

In [ ]:
data["entity_name"].value_counts()

In [ ]:
logs = pd.DataFrame(columns=["city_name", "entity_id", "input_tokens", "output_tokens", "costs", "mcp_usage"])
results = pd.DataFrame(columns=["city_name", "entity_id", "classification_result"])

for city in data[:1]["entity_name"].unique():
    city_data = data[data["entity_name"] == city]
    classification_base = [result["date"].strftime("%Y-%m-%d %H:%M:%S") + ":" + result["hits"] for _, result in city_data.iterrows()]
    city_data_point_count = len(city_data)
    city_name = city
    ris = city_data.iloc[0]['entity_id']
    user_prompt = f"Klassifiziere den Status der Wärmeplanung in {city_name}, mit der ID {ris}. Beziehe dich dabei auf folgende Daten und schau dir zuerst die jüngsten Ergebnisse an, da diese am wahrscheinlichsten eine Entscheidung enthalten. {classification_base}"
    response = await agent.run(user_prompt)
    
    # Tokens und MCP-Nutzung monitoren
    tokens_info = monitor_tokens(response)
    
    # In logs DataFrame speichern
    log_entry = {
        "city_name": city_name,
        "entity_id": ris,
        "input_tokens": tokens_info["input_tokens"],
        "output_tokens": tokens_info["output_tokens"],
        "costs": tokens_info["costs"],
        "mcp_usage": tokens_info["mcp_usage"]
    }
    logs = pd.concat([logs, pd.DataFrame([log_entry])], ignore_index=True)
    
    # In results DataFrame speichern
    result_entry = {
        "city_name": city_name,
        "entity_id": ris,
        "classification_result": response.output
    }
    results = pd.concat([results, pd.DataFrame([result_entry])], ignore_index=True)


In [ ]:
response.all_messages()

In [ ]:
logs

In [ ]:
results
